# Kepler 1 — Intent Classifier Training

This notebook loads the intents from `backend/chatbot/nlp/intents.json`, preprocesses the patterns with NLTK, trains a TF-IDF + Logistic Regression classifier and evaluates it.

Tech: Python, NLTK, scikit-learn.

In [ ]:
import json
import random
from pathlib import Path

import nltk
import numpy as np
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

for pkg in ('punkt', 'punkt_tab', 'wordnet', 'omw-1.4'):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

random.seed(42)
np.random.seed(42)

In [ ]:
INTENTS_PATH = Path('../backend/chatbot/nlp/intents.json').resolve()
with open(INTENTS_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f'Loaded {len(data["intents"])} intents from {INTENTS_PATH.name}')
for intent in data['intents']:
    print(f"  - {intent['tag']:<12} patterns={len(intent['patterns'])} responses={len(intent['responses'])}")

In [ ]:
lemmatizer = WordNetLemmatizer()

def normalize(text):
    tokens = word_tokenize(text.lower().strip())
    return ' '.join(lemmatizer.lemmatize(t) for t in tokens if t.isalpha())

texts, labels = [], []
for intent in data['intents']:
    for pattern in intent['patterns']:
        texts.append(normalize(pattern))
        labels.append(intent['tag'])

print(f'Total samples: {len(texts)}')
print('Sample after normalization:', texts[:5])

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ('clf', LogisticRegression(max_iter=1000, C=4.0)),
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, texts, labels, cv=cv, scoring='accuracy')
print(f'CV accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})')

pipeline.fit(texts, labels)
print('Trained on full dataset.')

In [ ]:
def predict(text, top_k=3):
    probs = pipeline.predict_proba([normalize(text)])[0]
    classes = pipeline.classes_
    order = np.argsort(probs)[::-1][:top_k]
    return [(classes[i], float(probs[i])) for i in order]

samples = [
    'hello there', 'thanks a lot', 'tell me a joke',
    'what tech stack do you use', 'who built you', 'see you tomorrow'
]
for s in samples:
    print(f'{s!r:<35} -> {predict(s)}')

In [ ]:
import joblib

MODEL_PATH = Path('../backend/chatbot/nlp/model.joblib').resolve()
responses = {i['tag']: i['responses'] for i in data['intents']}
joblib.dump({'pipeline': pipeline, 'responses': responses}, MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')